<div style="display:flex;gap:14px;align-items:center;flex-wrap:wrap;
 font-family:'Segoe UI',system-ui,sans-serif;font-size:13px;padding:10px 2px;
 border-bottom:2px solid #1B7A43;margin-bottom:4px;">
 <a href="https://colab.research.google.com/github/STG17-Africa/stg17-workshop/blob/main/notebooks/day4/D4_NTL_GEE_EN_open.ipynb" target="_blank"><img
  src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>
 <span style="color:#6B7B75;">Day 4 · open track</span>
 <span style="flex:1;"></span>
 <a href="./D4_NTL_GEE_FR_open.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">🌐 Français</a>
 <a href="./D4_NTL_GEE_EN.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">⇄ guided track</a>
</div>

<!-- Night-Time Lights on Google Earth Engine - nothing downloaded · STG17 workshop · AfDB / STATAFRIC -->
<!-- GENERATED FILE — edit notebooks/_masters/d4_ntl_gee.master.ipynb instead. -->


<div style="background:linear-gradient(135deg,#0B2545 0%,#1B7A43 100%);
 border-radius:18px;padding:32px 38px;font-family:'Segoe UI',system-ui,sans-serif;">
 <div style="color:#F2A900;font-size:12.5px;letter-spacing:3px;font-weight:700;
  text-transform:uppercase;">Day 4 · Earth Engine variant · runs in Colab</div>
 <div style="color:#fff;font-size:2em;font-weight:800;margin:10px 0 8px;line-height:1.15;">
  Night-Time Lights without downloading anything</div>
 <div style="color:#dbe7e0;font-size:1.05em;line-height:1.55;max-width:900px;">
  The same analysis as <code>D4_NTL_Collect_Explore</code>, with the pixels left where they are.
  You send an expression; Google runs it on their copy of the archive; a table comes back.
  A country-year statistic that costs 800 MB of download locally costs a few kilobytes here.
 </div>
 <div style="color:#F2A900;font-size:13px;margin-top:14px;font-weight:600;">
  Deliverable: the same national panel, plus a documented comparison against the local path.</div>
</div>


### When to use this notebook instead of the local one

Use Earth Engine when bandwidth is the constraint — which, on most institutional
networks in the region, it is. Use the local path when reproducibility on an
offline machine is the constraint, or when you need a product Google does not
host.

| | Local (rasterio) | Earth Engine |
|---|---|---|
| Network | none required | required |
| Who holds the pixels | you | Google |
| Reproducible offline | yes | no |
| Products available | anything NASA publishes | what Google has ingested |
| Scales with | your disk and RAM | Google's cluster |
| Long-term dependency | none | on a commercial service |

That last row is not a footnote. A national statistical office building a
production indicator on a free tier of a commercial service is taking on a
dependency it does not control — the sovereignty question of Day 1 afternoon, in
concrete form. The honest answer is usually: **prototype on Earth Engine,
produce on your own infrastructure**, and this notebook plus its local twin let
you do exactly that.


---
## Requirements and authorisation

Earth Engine needs a **free** account attached to a Google Cloud project.
Registration takes a few minutes at
[code.earthengine.google.com/register](https://code.earthengine.google.com/register)
and is on the pre-workshop checklist for exactly that reason.

The first `initialise()` opens a browser window once. After that, credentials are
cached.


In [ ]:
# --- Requirements for the Earth Engine variant -------------------------
REQUIREMENTS = {
    "ee":         "earthengine-api>=0.1.380",
    "geemap":     "geemap>=0.30",
    "pandas":     "pandas>=2.0",
    "matplotlib": "matplotlib>=3.7",
}

import subprocess
import sys

try:
    import stg17
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "stg17 @ git+https://github.com/STG17-Africa/stg17-workshop"], check=False)
    import stg17

from stg17 import setup, countries, theme, ui
from stg17.i18n import T

S = setup(REQUIREMENTS, lang="EN")

In [ ]:
# Your Cloud project id - from the Earth Engine registration page.
GEE_PROJECT = None   # EN: e.g. "ee-yourname". None lets Earth Engine pick a default. | FR: ex. "ee-votrenom". None laisse Earth Engine choisir par défaut.

from stg17 import ntl_gee

ee = ntl_gee.initialise(project=GEE_PROJECT)

---
## Step 1 — Your country

Same single variable as the local notebook. Nothing else changes.


In [ ]:
# ===========================================================================
# THE ONLY LINE YOU NEED TO CHANGE
# ===========================================================================
COUNTRY_ISO3 = "CIV"

YEARS         = list(range(2014, 2024))   # EN: EOG annual VNL covers 2012-2023 | FR: le VNL annuel EOG couvre 2012-2023
ADM_LEVEL     = 1
LIT_THRESHOLD = 0.5
SCALE_M       = 500       # EN: reduction resolution; raise to 1000 for a faster large country | FR: résolution de réduction ; passez à 1000 pour un grand pays plus rapide

C = countries.get(COUNTRY_ISO3)
OUT = S.outputs(C.iso3, "d4_ntl_gee")

nation = ntl_gee.country_geometry(ee, C, level=0)
zones = ntl_gee.country_geometry(ee, C, level=ADM_LEVEL)

n_zones = zones.size().getInfo()
print(f"{C.name('en')} ({C.iso3})")
print(T(f"  ADM{ADM_LEVEL} units found in FAO GAUL: {n_zones}",
        f"  Unités ADM{ADM_LEVEL} trouvées dans FAO GAUL : {n_zones}"))
print(T(f"  Reduction scale: {SCALE_M} m    Years: {YEARS[0]}-{YEARS[-1]}",
        f"  Échelle de réduction : {SCALE_M} m    Années : {YEARS[0]}-{YEARS[-1]}"))

if n_zones == 0:
    print(T("No units returned. GAUL spells some country names differently from ISO - "
            "check stg17.ntl_gee._GAUL_ALIASES, or upload your own boundaries as an asset.",
            "Aucune unité retournée. GAUL orthographie certains noms de pays différemment de l'ISO - "
            "vérifiez stg17.ntl_gee._GAUL_ALIASES, ou téléversez vos propres frontières comme asset."))

<div style="border:1px solid #F2A900;border-left:6px solid #F2A900;background:#FEF9EC;
 padding:13px 17px;border-radius:0 9px 9px 0;margin:14px 0;font-family:'Segoe UI',system-ui,sans-serif;">
<b style="color:#F2A900;font-size:11.5px;letter-spacing:1.6px;">▲ WATCH OUT — FAO GAUL IS FROM 2015</b><br>
<span style="color:#33403A;font-size:14.3px;line-height:1.58;">
The boundaries built into Earth Engine are the FAO GAUL 2015 release. They predate several
administrative reorganisations across the continent, and they still call Eswatini "Swaziland".
They are fine for a prototype. For anything you publish, upload your national boundary file as an
Earth Engine asset and pass it via <code>asset=</code> — the function supports it, and Day 5
explains the workflow.
</span></div>


---
## Step 2 — Look at one year, interactively

`geemap` renders an Earth Engine image as a live tile layer. Pan and zoom: the
tiles are computed on demand, so exploring a whole country costs nothing more
than exploring one district.

Note the colour ramp — it is the same `stg17_night` ramp the local notebook uses,
so a map produced here and a map produced there are directly comparable.


In [ ]:
# TODO: Build an interactive map for the most recent year with ntl_gee.map_year()
...

---
## Step 3 — The panel: year × administrative unit

This is the same table the local notebook builds, computed server-side. What
comes back over the network is a few hundred rows of numbers.

Watch the timing. On a normal connection this is faster than downloading a single
granule, and it covers ten years.


In [ ]:
# TODO: Call ntl_gee.zonal_panel() over YEARS and time it
...

---
## Step 4 — The national series

Two panels: the national Sum of Lights indexed to its first year, and the share
of territory above the lit threshold. Together they separate "more light in the
same places" from "light reaching new places" — a distinction that matters
enormously for an electrification reading and that a single series hides.


In [ ]:
# TODO: Aggregate the panel to national level and plot the two series
...

---
## Step 5 — Compare the two pipelines

This is the step that makes the notebook worth running even if you already did
the local one.

Load the panel your local notebook produced and put the two national series side
by side. **They will not be identical.** EOG's annual VNL and NASA's VNP46A4 use
different compositing, different outlier removal and different masking. A gap of
a few percent is expected and healthy; a gap of a factor of two means one of the
two runs has a problem worth finding.

What a statistical office takes from this: the number depends on the pipeline,
so the pipeline is part of the metadata. Publishing "Sum of Lights = 4.2 million"
without naming the product and the version is not a reproducible statistic.


In [ ]:
# TODO: Load the local panel if present and overlay the two indexed national series
...

---
## Step 6 — Export and save

Two outputs. The panel as CSV, which is what the afternoon laboratory consumes.
And, optionally, a clipped GeoTIFF exported to your Drive — the practical bridge
between the two worlds: composite on Google's cluster, then continue in rasterio
on one manageable national file instead of six 10° tiles.


In [ ]:
# TODO: Save the panel to CSV with a metadata sidecar, then optionally queue a Drive export
...

---
## Limitations of this path specifically

<div style="border:1px solid #F2A900;border-radius:10px;background:#FFFDF6;
 padding:16px 20px;margin:16px 0;font-family:'Segoe UI',system-ui,sans-serif;">
<ul style="color:#33403A;font-size:14px;line-height:1.7;margin:0;padding-left:22px;">
<li><b>All the limitations of the local notebook still apply</b> — proxy status, threshold
sensitivity, gas flares, saturation, boundary choice. Earth Engine changes how you compute, not
what the light means.</li>
<li><b>Different product, different numbers.</b> EOG annual VNL is not NASA VNP46A4. Name the one
you used in every published figure.</li>
<li><b>FAO GAUL 2015 boundaries</b> are the default here and are not your official boundaries.</li>
<li><b>The reduction scale is a parameter.</b> Reducing at 1000 m instead of 500 m is four times
faster and quietly loses small settlements. Whichever you choose, record it.</li>
<li><b>getInfo() is capped at 5000 features.</b> A country with more ADM2 units than that needs
the panel built in groups, or exported to Drive as a table.</li>
<li><b>Continuity risk.</b> This result is reproducible only while Google serves the collection
under terms you can accept. For a production statistical indicator, that is a governance question
to answer before the pipeline is adopted, not after.</li>
</ul></div>

<div style="background:linear-gradient(135deg,#0B2545,#1B7A43);border-radius:16px;
 padding:22px 30px;margin-top:22px;text-align:center;font-family:'Segoe UI',system-ui,sans-serif;">
 <div style="color:#F2A900;font-size:11.5px;letter-spacing:2.5px;font-weight:700;">
  DATA SCIENCE TOOLKIT · AfDB / STATAFRIC</div>
 <div style="color:#fff;font-size:1.05em;margin-top:7px;font-weight:600;">
  STG17 workshop · Action Plan 2025-2030 · activities 4.2.1, 4.2.3 and 2.1.1</div></div>
